# RNN/LSTM Demo Notebook

Demo cepat RNN/LSTM untuk memastikan seluruh cell selesai tanpa eksperimen berat. Default-nya `RUN_MODE = "demo"` dan memakai synthetic feature cache kecil, sehingga cocok untuk sanity check sebelum menjalankan `rnn_lstm.ipynb` mode full di Kaggle.

## 0. Setup

In [ ]:
import csv
import json
import os
import random
import shutil
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = next(parent for parent in Path.cwd().parents if (parent / "src").exists())

RNN_SRC = REPO_ROOT / "src" / "rnn-lstm"
UTILS_SRC = REPO_ROOT / "src" / "utils"
for path in (str(REPO_ROOT), str(RNN_SRC), str(UTILS_SRC)):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Repo root:", REPO_ROOT)
print("TensorFlow:", tf.__version__)

In [ ]:
from feature_extraction import extract_flickr8k_features, load_features
from text_utils import (
    clean_caption,
    decode_caption,
    encode_caption,
    load_flickr8k_captions,
    load_vocab,
)
import preprocess_flickr8k
from decoder_model import build_dataset, build_decoder_model, masked_accuracy, masked_loss
from train_decoder import DecoderExperimentConfig, train_one
from caption_model import CaptionModel
from evaluate_captioning import evaluate
from metrics import bleu_score, meteor_score, tokenize
from rnn import SimpleRNNCell, SimpleRNNDecoder
from lstm import LSTMCell, LSTMDecoder

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IS_KAGGLE = Path("/kaggle/input").exists()


def env_flag(name: str, default: bool) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


RUN_MODE = os.getenv("RUN_MODE", "demo").strip().lower()
if RUN_MODE == "smoke":
    RUN_MODE = "demo"
if RUN_MODE not in {"local", "demo", "full"}:
    raise ValueError("RUN_MODE harus 'local', 'demo', atau 'full'")

DEMO_MODE = RUN_MODE == "demo"
DEMO_SYNTHETIC_FEATURES = env_flag("DEMO_SYNTHETIC_FEATURES", DEMO_MODE)

RUN_EXTRACT_FEATURES = env_flag("RUN_EXTRACT_FEATURES", IS_KAGGLE and not DEMO_SYNTHETIC_FEATURES)
RUN_PREPROCESS = env_flag("RUN_PREPROCESS", IS_KAGGLE or DEMO_MODE)
RUN_TRAIN_RNN = env_flag("RUN_TRAIN_RNN", IS_KAGGLE or DEMO_MODE)
RUN_TRAIN_LSTM = env_flag("RUN_TRAIN_LSTM", IS_KAGGLE or DEMO_MODE)
RUN_TRAIN_INIT_INJECT = env_flag("RUN_TRAIN_INIT_INJECT", IS_KAGGLE and RUN_MODE == "full")
RUN_FULL_EVAL = env_flag("RUN_FULL_EVAL", IS_KAGGLE or DEMO_MODE)

N_EVAL = int(os.getenv("N_EVAL", "5" if DEMO_MODE else "50"))
N_BATCH = int(os.getenv("N_BATCH", str(N_EVAL)))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "8" if DEMO_MODE else "64"))
EPOCHS = int(os.getenv("EPOCHS", "1" if DEMO_MODE else "5"))
MAX_SEQ_LEN = int(os.getenv("MAX_SEQ_LEN", "20" if DEMO_MODE else "35"))
EMBED_DIM = int(os.getenv("EMBED_DIM", "64" if DEMO_MODE else "256"))
FEATURE_DIM = int(os.getenv("FEATURE_DIM", "2048"))
LEARNING_RATE = float(os.getenv("LEARNING_RATE", "1e-3"))

DEMO_FLICKR_TRAIN_IMAGES = int(os.getenv("DEMO_FLICKR_TRAIN_IMAGES", "18"))
DEMO_FLICKR_VAL_IMAGES = int(os.getenv("DEMO_FLICKR_VAL_IMAGES", "4"))
DEMO_FLICKR_TEST_IMAGES = int(os.getenv("DEMO_FLICKR_TEST_IMAGES", "4"))

FULL_VARIATIONS = [
    {"n_layers": 1, "hidden_size": 128},
    {"n_layers": 1, "hidden_size": 256},
    {"n_layers": 2, "hidden_size": 128},
    {"n_layers": 2, "hidden_size": 256},
    {"n_layers": 3, "hidden_size": 128},
    {"n_layers": 3, "hidden_size": 256},
]
DEMO_VARIATIONS = [{"n_layers": 1, "hidden_size": 64}]
VARIATIONS = DEMO_VARIATIONS if DEMO_MODE else FULL_VARIATIONS

{
    "RUN_MODE": RUN_MODE,
    "IS_KAGGLE": IS_KAGGLE,
    "DEMO_SYNTHETIC_FEATURES": DEMO_SYNTHETIC_FEATURES,
    "RUN_EXTRACT_FEATURES": RUN_EXTRACT_FEATURES,
    "RUN_PREPROCESS": RUN_PREPROCESS,
    "RUN_TRAIN_RNN": RUN_TRAIN_RNN,
    "RUN_TRAIN_LSTM": RUN_TRAIN_LSTM,
    "RUN_FULL_EVAL": RUN_FULL_EVAL,
    "EPOCHS": EPOCHS,
    "VARIATIONS": VARIATIONS,
}

## 0a. Path Resolution

In [ ]:
FLICKR8K_DATASET_SLUG = os.getenv("FLICKR8K_DATASET_SLUG", "").strip()
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else REPO_ROOT / "outputs"
DEFAULT_WORKING_DIR = (
    WORK_ROOT / "outputs_demo" / "rnn_lstm" if IS_KAGGLE and DEMO_MODE else
    WORK_ROOT / "outputs" / "rnn_lstm" if IS_KAGGLE else
    WORK_ROOT / ("rnn_lstm_demo" if DEMO_MODE else "rnn_lstm")
)
WORKING_DIR = Path(os.getenv("WORKING_DIR", DEFAULT_WORKING_DIR))
FEATURES_PATH = WORKING_DIR / "features.npy"
PREP_DIR = WORKING_DIR / "preprocessed"
VOCAB_PATH = PREP_DIR / "vocab.json"
SPLITS_JSON = PREP_DIR / "splits.json"
MODELS_DIR = WORKING_DIR / "models"
EVAL_DIR = WORKING_DIR / "eval"
PLOTS_DIR = WORKING_DIR / "plots"

for p in (WORKING_DIR, PREP_DIR, MODELS_DIR, EVAL_DIR, PLOTS_DIR):
    p.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def optional_path(value):
    if value in (None, ""):
        return None
    return Path(value)


def path_from_slug(slug: str):
    if not IS_KAGGLE or not slug:
        return None
    path = INPUT_ROOT / slug
    return path if path.exists() else None


def find_file(name_options, preferred_slug=""):
    roots = []
    explicit = path_from_slug(preferred_slug)
    if explicit:
        roots.append(explicit)
    if IS_KAGGLE and INPUT_ROOT.exists():
        roots.append(INPUT_ROOT)
    for root in roots:
        for name in name_options:
            matches = sorted(root.glob(f"**/{name}"))
            if matches:
                return matches[0]
    return None


def find_dir(name_options, preferred_slug=""):
    roots = []
    explicit = path_from_slug(preferred_slug)
    if explicit:
        roots.append(explicit)
    if IS_KAGGLE and INPUT_ROOT.exists():
        roots.append(INPUT_ROOT)
    for root in roots:
        for name in name_options:
            matches = sorted(path for path in root.glob(f"**/{name}") if path.is_dir())
            if matches:
                return matches[0]
    return None


def caption_image_names(path: Path):
    names = []
    if not path or not path.exists():
        return names
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line or line.lower().startswith("image,caption"):
                continue
            left = line.split("\t", 1)[0] if "\t" in line else line.split(",", 1)[0]
            name = left.split("#", 1)[0].strip()
            if name:
                names.append(name)
    return sorted(set(names))


def write_split(path: Path, names):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(names) + "\n", encoding="utf-8")


def image_map(image_dir: Path):
    if image_dir is None or not image_dir.exists():
        return {}
    return {p.name: p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS}


def discover_flickr8k_paths():
    env_root = optional_path(os.getenv("FLICKR8K_ROOT"))
    env_images = optional_path(os.getenv("FLICKR8K_IMAGES"))
    env_captions = optional_path(os.getenv("FLICKR8K_CAPTIONS"))
    env_train = optional_path(os.getenv("FLICKR8K_TRAIN_SPLIT"))
    env_val = optional_path(os.getenv("FLICKR8K_VAL_SPLIT"))
    env_test = optional_path(os.getenv("FLICKR8K_TEST_SPLIT"))

    if IS_KAGGLE:
        root = env_root or path_from_slug(FLICKR8K_DATASET_SLUG) or find_dir(["flickr8k", "Flickr8k"], FLICKR8K_DATASET_SLUG)
        images = env_images or find_dir(["Images", "Flicker8k_Dataset", "Flickr8k_Dataset"], FLICKR8K_DATASET_SLUG)
        captions = env_captions or find_file(["captions.txt", "Flickr8k.token.txt"], FLICKR8K_DATASET_SLUG)
        train = env_train or find_file(["Flickr_8k.trainImages.txt", "Flickr8k.trainImages.txt"], FLICKR8K_DATASET_SLUG)
        val = env_val or find_file(["Flickr_8k.devImages.txt", "Flickr8k.devImages.txt"], FLICKR8K_DATASET_SLUG)
        test = env_test or find_file(["Flickr_8k.testImages.txt", "Flickr8k.testImages.txt"], FLICKR8K_DATASET_SLUG)
    else:
        root = env_root or REPO_ROOT / "data" / "flickr8k"
        images = env_images or root / "Images"
        captions = env_captions or root / "captions.txt"
        train = env_train or root / "Flickr_8k.trainImages.txt"
        val = env_val or root / "Flickr_8k.devImages.txt"
        test = env_test or root / "Flickr_8k.testImages.txt"
    return root, images, captions, train, val, test


FLICKR_INPUT, IMAGE_DIR, CAPTIONS_PATH, TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT = discover_flickr8k_paths()


def create_synthetic_demo_dataset():
    rng = np.random.default_rng(SEED)
    demo_dir = WORKING_DIR / "synthetic_flickr8k"
    image_dir = demo_dir / "Images"
    image_dir.mkdir(parents=True, exist_ok=True)
    total = DEMO_FLICKR_TRAIN_IMAGES + DEMO_FLICKR_VAL_IMAGES + DEMO_FLICKR_TEST_IMAGES
    names = [f"demo_{i:03d}.jpg" for i in range(total)]

    try:
        from PIL import Image
        for i, name in enumerate(names):
            arr = np.zeros((64, 64, 3), dtype=np.uint8)
            arr[..., 0] = (37 * i) % 255
            arr[..., 1] = (83 * i) % 255
            arr[..., 2] = (127 * i) % 255
            Image.fromarray(arr).save(image_dir / name)
    except Exception as exc:
        print("Synthetic image write skipped:", exc)

    templates = [
        "a person stands near a small dog",
        "a child plays outside on the grass",
        "two people walk beside the water",
        "a group sits near a bright street",
        "a dog runs through a green field",
    ]
    captions_path = demo_dir / "captions.txt"
    with captions_path.open("w", encoding="utf-8") as handle:
        handle.write("image,caption\n")
        for i, name in enumerate(names):
            handle.write(f"{name},{templates[i % len(templates)]}\n")
            handle.write(f"{name},{templates[(i + 1) % len(templates)]}\n")

    split_dir = demo_dir / "splits"
    train_names = names[:DEMO_FLICKR_TRAIN_IMAGES]
    val_names = names[DEMO_FLICKR_TRAIN_IMAGES:DEMO_FLICKR_TRAIN_IMAGES + DEMO_FLICKR_VAL_IMAGES]
    test_names = names[DEMO_FLICKR_TRAIN_IMAGES + DEMO_FLICKR_VAL_IMAGES:]
    train_path = split_dir / "Flickr_8k.trainImages.txt"
    val_path = split_dir / "Flickr_8k.devImages.txt"
    test_path = split_dir / "Flickr_8k.testImages.txt"
    write_split(train_path, train_names)
    write_split(val_path, val_names)
    write_split(test_path, test_names)

    features = {name: rng.normal(0.0, 0.25, size=(FEATURE_DIM,)).astype(np.float32) for name in names}
    np.save(FEATURES_PATH, features)
    print("Synthetic demo dataset:", demo_dir)
    print("Synthetic cached features:", FEATURES_PATH)
    return demo_dir, image_dir, captions_path, train_path, val_path, test_path


def make_real_demo_subset(image_dir: Path, captions_path: Path):
    lookup = image_map(image_dir)
    caption_names = [name for name in caption_image_names(captions_path) if name in lookup]
    if not caption_names:
        caption_names = sorted(lookup)
    need = DEMO_FLICKR_TRAIN_IMAGES + DEMO_FLICKR_VAL_IMAGES + DEMO_FLICKR_TEST_IMAGES
    selected = caption_names[:need]
    if len(selected) < 3:
        raise RuntimeError("Demo Flickr8k gagal: gambar/caption yang cocok terlalu sedikit.")

    demo_images = WORKING_DIR / "demo_images"
    if demo_images.exists():
        shutil.rmtree(demo_images)
    demo_images.mkdir(parents=True, exist_ok=True)
    for name in selected:
        shutil.copy2(lookup[name], demo_images / name)

    split_dir = WORKING_DIR / "demo_splits"
    train_names = selected[:DEMO_FLICKR_TRAIN_IMAGES]
    val_names = selected[DEMO_FLICKR_TRAIN_IMAGES:DEMO_FLICKR_TRAIN_IMAGES + DEMO_FLICKR_VAL_IMAGES]
    test_names = selected[DEMO_FLICKR_TRAIN_IMAGES + DEMO_FLICKR_VAL_IMAGES:]
    if not val_names:
        val_names = train_names[-1:]
    if not test_names:
        test_names = val_names[-1:]
    train_path = split_dir / "Flickr_8k.trainImages.txt"
    val_path = split_dir / "Flickr_8k.devImages.txt"
    test_path = split_dir / "Flickr_8k.testImages.txt"
    write_split(train_path, train_names)
    write_split(val_path, val_names)
    write_split(test_path, test_names)
    print("Real demo Flickr images:", len(selected), demo_images)
    return demo_images, train_path, val_path, test_path


if DEMO_MODE and DEMO_SYNTHETIC_FEATURES:
    FLICKR_INPUT, IMAGE_DIR, CAPTIONS_PATH, TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT = create_synthetic_demo_dataset()
    RUN_EXTRACT_FEATURES = False
elif DEMO_MODE and IMAGE_DIR is not None and CAPTIONS_PATH is not None and IMAGE_DIR.exists() and CAPTIONS_PATH.exists():
    IMAGE_DIR, TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT = make_real_demo_subset(IMAGE_DIR, CAPTIONS_PATH)

if CAPTIONS_PATH is not None and CAPTIONS_PATH.exists() and not all(
    p is not None and p.exists() for p in (TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT)
):
    split_dir = WORKING_DIR / "generated_splits"
    names = caption_image_names(CAPTIONS_PATH)
    if not names and IMAGE_DIR is not None:
        names = sorted(image_map(IMAGE_DIR))
    if len(names) < 3:
        raise RuntimeError("Tidak bisa generate split Flickr8k: nama gambar terlalu sedikit atau captions tidak terbaca.")
    train_names = names[: min(6000, max(1, int(0.75 * len(names))))]
    val_start = len(train_names)
    val_end = min(val_start + 1000, val_start + max(1, int(0.125 * len(names))))
    val_names = names[val_start:val_end]
    test_names = names[val_end:]
    if not test_names:
        test_names = val_names[-max(1, len(val_names) // 2):]
        val_names = val_names[:-len(test_names)] or test_names
    TRAIN_SPLIT = split_dir / "Flickr_8k.trainImages.txt"
    VAL_SPLIT = split_dir / "Flickr_8k.devImages.txt"
    TEST_SPLIT = split_dir / "Flickr_8k.testImages.txt"
    write_split(TRAIN_SPLIT, train_names)
    write_split(VAL_SPLIT, val_names)
    write_split(TEST_SPLIT, test_names)
    print("Generated Flickr8k splits:", len(train_names), len(val_names), len(test_names))

for key, value in {
    "FLICKR8K_ROOT": FLICKR_INPUT,
    "FLICKR8K_IMAGES": IMAGE_DIR,
    "FLICKR8K_CAPTIONS": CAPTIONS_PATH,
    "FLICKR8K_TRAIN_SPLIT": TRAIN_SPLIT,
    "FLICKR8K_VAL_SPLIT": VAL_SPLIT,
    "FLICKR8K_TEST_SPLIT": TEST_SPLIT,
    "WORKING_DIR": WORKING_DIR,
}.items():
    if value is not None:
        os.environ[key] = str(value)

{
    "RUN_MODE": RUN_MODE,
    "IMAGE_DIR": str(IMAGE_DIR),
    "CAPTIONS_PATH": str(CAPTIONS_PATH),
    "TRAIN_SPLIT": str(TRAIN_SPLIT),
    "VAL_SPLIT": str(VAL_SPLIT),
    "TEST_SPLIT": str(TEST_SPLIT),
    "FEATURES_PATH": str(FEATURES_PATH),
    "VOCAB_PATH": str(VOCAB_PATH),
    "MODELS_DIR": str(MODELS_DIR),
}

In [ ]:
def read_json(path: Path, default=None):
    if not path.exists():
        return default
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2)


def load_splits(path: Path):
    data = read_json(path, {"train": [], "val": [], "test": []})
    return {"train": data.get("train", []), "val": data.get("val", []), "test": data.get("test", [])}


def has_required_files():
    required = [CAPTIONS_PATH, VOCAB_PATH, SPLITS_JSON, FEATURES_PATH]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        print("File belum tersedia:")
        for p in missing:
            print(" -", p)
        return False
    return True

## 1. CNN Encoder Frozen

In [ ]:
if RUN_EXTRACT_FEATURES:
    if not IMAGE_DIR.exists():
        raise FileNotFoundError(IMAGE_DIR)
    features = extract_flickr8k_features(
        images_dir=str(IMAGE_DIR),
        output_path=str(FEATURES_PATH),
        encoder_name="inceptionv3",
        batch_size=32,
    )
    print("Extracted features:", len(features))
elif FEATURES_PATH.exists():
    features_preview = load_features(str(FEATURES_PATH))
    print("Loaded cached features:", len(features_preview))
else:
    print("Feature cache belum ada. Set RUN_EXTRACT_FEATURES=True di Kaggle untuk membuat features.npy.")

In [ ]:
def get_image_feature(img_path: Path):
    from tensorflow.keras.applications import InceptionV3
    from tensorflow.keras.applications.inception_v3 import preprocess_input
    from tensorflow.keras.preprocessing.image import img_to_array, load_img

    encoder = InceptionV3(weights="imagenet", include_top=False, pooling="avg")
    image = load_img(img_path, target_size=(299, 299))
    arr = img_to_array(image)[np.newaxis]
    return encoder.predict(preprocess_input(arr), verbose=0)[0].astype(np.float32)

## 2. Caption Preprocessing

In [ ]:
if RUN_PREPROCESS:
    summary = preprocess_flickr8k.preprocess(
        captions_file=CAPTIONS_PATH,
        train_split=TRAIN_SPLIT,
        val_split=VAL_SPLIT,
        test_split=TEST_SPLIT,
        output_dir=PREP_DIR,
        min_freq=1,
    )
    summary
else:
    summary = read_json(PREP_DIR / "summary.json", {})
    summary

In [ ]:
if CAPTIONS_PATH.exists():
    captions = load_flickr8k_captions(str(CAPTIONS_PATH))
    print("Total image captions:", len(captions))
    first_key = next(iter(captions), None)
    if first_key:
        print(first_key, "->", captions[first_key][:2])
else:
    captions = {}
    print("captions.txt belum ditemukan.")

In [ ]:
if VOCAB_PATH.exists():
    word2idx, idx2word = load_vocab(VOCAB_PATH)
    vocab_size = len(word2idx)
    START_IDX = word2idx["<start>"]
    END_IDX = word2idx["<end>"]
    PAD_IDX = word2idx["<pad>"]
    print("Vocab size:", vocab_size)
    print("Special tokens:", {"pad": PAD_IDX, "start": START_IDX, "end": END_IDX})
else:
    word2idx, idx2word, vocab_size = {}, {}, 0
    print("vocab.json belum ada.")

In [ ]:
if captions and word2idx:
    lengths = [len(encode_caption(cap, word2idx, max_len=None)) for caps in captions.values() for cap in caps]
    print("Caption length min/mean/p95/max:", int(np.min(lengths)), float(np.mean(lengths)), int(np.percentile(lengths, 95)), int(np.max(lengths)))
    plt.figure(figsize=(6, 3))
    plt.hist(lengths, bins=30)
    plt.xlabel("caption length incl. special tokens")
    plt.ylabel("count")
    plt.tight_layout()
    plt.show()

## 3. Dataset Builder Smoke Test

In [ ]:
if has_required_files():
    splits = load_splits(SPLITS_JSON)
    features = load_features(str(FEATURES_PATH))
    train_names = [name for name in splits["train"] if name in captions and name in features][:8]
    ds_preview = build_dataset(
        train_names,
        captions,
        features,
        word2idx,
        max_seq_len=MAX_SEQ_LEN,
        batch_size=4,
        shuffle=False,
        injection_method="pre",
    )
    for (img_batch, tok_batch), y_batch in ds_preview.take(1):
        print("image_features", img_batch.shape)
        print("caption_tokens", tok_batch.shape)
        print("target", y_batch.shape)
else:
    splits, features = {"train": [], "val": [], "test": []}, {}

## 4. RNN Decoder Training

In [ ]:
def make_config(decoder_type, n_layers, hidden_size, injection_method="pre"):
    return DecoderExperimentConfig(
        decoder_type=decoder_type,
        n_layers=n_layers,
        hidden_size=hidden_size,
        embed_dim=EMBED_DIM,
        max_seq_len=MAX_SEQ_LEN,
        feature_dim=FEATURE_DIM,
        learning_rate=LEARNING_RATE,
        injection_method=injection_method,
    )


def decoder_configs(decoder_type, injection_method="pre"):
    return [make_config(decoder_type, **variation, injection_method=injection_method) for variation in VARIATIONS]

rnn_configs = decoder_configs("rnn")
[c.experiment_id for c in rnn_configs]

In [ ]:
def train_grid(configs, run_flag):
    trained, skipped = [], []
    if not run_flag:
        print("Training flag False; hanya cek artefak yang sudah ada.")
    if not has_required_files():
        return trained, skipped

    splits = load_splits(SPLITS_JSON)
    features = load_features(str(FEATURES_PATH))
    for config in configs:
        exp_dir = MODELS_DIR / config.experiment_id
        if (exp_dir / "model.keras").exists():
            skipped.append(config.experiment_id)
            continue
        if run_flag:
            print("Training", config.experiment_id)
            train_one(config, captions, splits, features, word2idx, MODELS_DIR, BATCH_SIZE, EPOCHS, SEED)
            trained.append(config.experiment_id)
        else:
            skipped.append(config.experiment_id)
    return trained, skipped

In [ ]:
rnn_trained, rnn_skipped = train_grid(rnn_configs, RUN_TRAIN_RNN)
{"trained": rnn_trained, "available_or_skipped": rnn_skipped}

In [ ]:
def history_summary(experiment_id):
    exp_dir = MODELS_DIR / experiment_id
    history = read_json(exp_dir / "history.json", {})
    metrics = read_json(exp_dir / "metrics.json", {})
    config = read_json(exp_dir / "config.json", {})
    if not history and not metrics:
        return None
    return {
        "experiment": experiment_id,
        "decoder_type": config.get("decoder_type"),
        "injection": config.get("injection_method"),
        "layers": config.get("n_layers"),
        "hidden": config.get("hidden_size"),
        "loss": (history.get("loss") or [None])[-1],
        "val_loss": (history.get("val_loss") or [None])[-1],
        "param_count": metrics.get("param_count"),
    }


def summarize_decoder(decoder_type, injection="pre"):
    rows = []
    for config in decoder_configs(decoder_type, injection):
        row = history_summary(config.experiment_id)
        if row:
            rows.append(row)
    rows.sort(key=lambda r: float("inf") if r["val_loss"] is None else r["val_loss"])
    return rows

rnn_summary = summarize_decoder("rnn")
rnn_summary

In [ ]:
def plot_histories(rows, title, output_path=None):
    if not rows:
        print("Belum ada history untuk diplot.")
        return
    cols = 3
    rows_n = int(np.ceil(len(rows) / cols))
    plt.figure(figsize=(cols * 4, rows_n * 3))
    for i, row in enumerate(rows, start=1):
        hist = read_json(MODELS_DIR / row["experiment"] / "history.json", {})
        ax = plt.subplot(rows_n, cols, i)
        ax.plot(hist.get("loss", []), label="train")
        ax.plot(hist.get("val_loss", []), label="val")
        ax.set_title(row["experiment"], fontsize=9)
        ax.set_xlabel("epoch")
        ax.set_ylabel("loss")
        ax.legend(fontsize=8)
    plt.suptitle(title)
    plt.tight_layout()
    if output_path:
        plt.savefig(output_path)
    plt.show()

plot_histories(rnn_summary, "RNN training loss", PLOTS_DIR / "rnn_loss_curves.png")

## 5. LSTM Decoder Training

In [ ]:
lstm_configs = decoder_configs("lstm")
[c.experiment_id for c in lstm_configs]

In [ ]:
lstm_trained, lstm_skipped = train_grid(lstm_configs, RUN_TRAIN_LSTM)
{"trained": lstm_trained, "available_or_skipped": lstm_skipped}

In [ ]:
lstm_summary = summarize_decoder("lstm")
lstm_summary

In [ ]:
plot_histories(lstm_summary, "LSTM training loss", PLOTS_DIR / "lstm_loss_curves.png")

## 6. Select Best Models

In [ ]:
def select_best(rows):
    if not rows:
        return None
    return min(rows, key=lambda r: float("inf") if r["val_loss"] is None else r["val_loss"])

best_rnn = select_best(rnn_summary)
best_lstm = select_best(lstm_summary)
best_rnn, best_lstm

In [ ]:
def load_keras_caption_model(experiment_id):
    model_path = MODELS_DIR / experiment_id / "model.keras"
    if not model_path.exists():
        raise FileNotFoundError(model_path)
    # Inference dan scratch weight extraction tidak membutuhkan compile state.
    # compile=False juga menghindari error deserialisasi Keras 3 untuk @tf.function masked_loss.
    return tf.keras.models.load_model(model_path, compile=False)


def load_scratch_caption_model(experiment_id, decoder_type, injection="pre"):
    keras_model = load_keras_caption_model(experiment_id)
    scratch = CaptionModel(decoder_type=decoder_type, injection_method=injection)
    scratch.load_from_keras(keras_model, word2idx, idx2word)
    return scratch

## 7. Evaluation Helpers

In [ ]:
def filtered_eval_names(split="test", n=N_EVAL):
    if not has_required_files():
        return []
    splits = load_splits(SPLITS_JSON)
    feats = load_features(str(FEATURES_PATH))
    names = [name for name in splits[split] if name in captions and name in feats]
    return names[:n] if n else names


def evaluate_with_time(model, decoder_type, mode, image_names, max_len=MAX_SEQ_LEN, decoding="greedy", beam_size=3, injection="pre"):
    feats = load_features(str(FEATURES_PATH))
    start = time.perf_counter()
    result = evaluate(
        model=model,
        captions=captions,
        image_names=image_names,
        features=feats,
        word2idx=word2idx,
        idx2word=idx2word,
        decoder_type=decoder_type,
        max_len=max_len,
        mode=mode,
        seed=SEED,
        decoding=decoding,
        beam_size=beam_size,
        length_penalty=0.7,
    )
    elapsed = time.perf_counter() - start
    result["metrics"]["total_time_s"] = elapsed
    result["metrics"]["avg_time_ms"] = 1000 * elapsed / max(len(result["details"]), 1)
    result["metrics"]["decoding"] = decoding
    result["metrics"]["beam_size"] = beam_size if decoding == "beam" else None
    result["metrics"]["injection"] = injection
    return result

In [ ]:
def evaluate_experiment(experiment_id, decoder_type, image_names, modes=("keras", "scratch"), decoding="greedy", beam_size=3, injection="pre"):
    out = {}
    if "keras" in modes:
        keras_model = load_keras_caption_model(experiment_id)
        out["keras"] = evaluate_with_time(keras_model, decoder_type, "keras", image_names, decoding=decoding, beam_size=beam_size, injection=injection)
    if "scratch" in modes:
        scratch_model = load_scratch_caption_model(experiment_id, decoder_type, injection=injection)
        out["scratch"] = evaluate_with_time(scratch_model, decoder_type, "scratch", image_names, decoding=decoding, beam_size=beam_size, injection=injection)
    return out

## 8. RNN Evaluation: Keras vs Scratch

In [ ]:
eval_names = filtered_eval_names("test", N_EVAL)
print("Eval samples:", len(eval_names))

rnn_best_eval = {}
if best_rnn and eval_names and RUN_FULL_EVAL:
    rnn_best_eval = evaluate_experiment(best_rnn["experiment"], "rnn", eval_names, modes=("keras", "scratch"))
    write_json(EVAL_DIR / "best_rnn_eval.json", rnn_best_eval)
else:
    rnn_best_eval = read_json(EVAL_DIR / "best_rnn_eval.json", {})

{k: v.get("metrics", {}) for k, v in rnn_best_eval.items()}

## 9. LSTM Evaluation: Keras vs Scratch

In [ ]:
lstm_best_eval = {}
if best_lstm and eval_names and RUN_FULL_EVAL:
    lstm_best_eval = evaluate_experiment(best_lstm["experiment"], "lstm", eval_names, modes=("keras", "scratch"))
    write_json(EVAL_DIR / "best_lstm_eval.json", lstm_best_eval)
else:
    lstm_best_eval = read_json(EVAL_DIR / "best_lstm_eval.json", {})

{k: v.get("metrics", {}) for k, v in lstm_best_eval.items()}

## 10. Evaluate All Variations

In [ ]:
def evaluate_all_variations(decoder_type, rows, max_samples=N_EVAL):
    names = filtered_eval_names("test", max_samples)
    results = {}
    for row in rows:
        exp_id = row["experiment"]
        model_path = MODELS_DIR / exp_id / "model.keras"
        if not model_path.exists():
            continue
        print("Evaluating", exp_id)
        keras_model = load_keras_caption_model(exp_id)
        res = evaluate_with_time(keras_model, decoder_type, "keras", names)
        results[exp_id] = {
            "summary": res["metrics"],
            "results": res["details"],
            "num_layers": row["layers"],
            "hidden_units": row["hidden"],
        }
    return results

if RUN_FULL_EVAL:
    rnn_variation_results = evaluate_all_variations("rnn", rnn_summary)
    lstm_variation_results = evaluate_all_variations("lstm", lstm_summary)
    write_json(EVAL_DIR / "rnn_variations.json", rnn_variation_results)
    write_json(EVAL_DIR / "lstm_variations.json", lstm_variation_results)
else:
    rnn_variation_results = read_json(EVAL_DIR / "rnn_variations.json", {})
    lstm_variation_results = read_json(EVAL_DIR / "lstm_variations.json", {})

len(rnn_variation_results), len(lstm_variation_results)

In [ ]:
def variation_table(results):
    rows = []
    for name, payload in results.items():
        summary = payload.get("summary", {})
        rows.append({
            "experiment": name,
            "layers": payload.get("num_layers"),
            "hidden": payload.get("hidden_units"),
            "bleu4": summary.get("bleu4"),
            "meteor": summary.get("meteor"),
            "avg_time_ms": summary.get("avg_time_ms"),
        })
    rows.sort(key=lambda r: r.get("bleu4") or -1, reverse=True)
    return rows

rnn_variation_table = variation_table(rnn_variation_results)
lstm_variation_table = variation_table(lstm_variation_results)
rnn_variation_table[:3], lstm_variation_table[:3]

In [ ]:
def plot_variation_bleu(rnn_rows, lstm_rows):
    rows = [("RNN", r) for r in rnn_rows] + [("LSTM", r) for r in lstm_rows]
    if not rows:
        print("Belum ada hasil variasi untuk diplot.")
        return
    labels = [f"{kind}\nL{r['layers']} H{r['hidden']}" for kind, r in rows]
    bleu = [r.get("bleu4") or 0 for _, r in rows]
    meteor = [r.get("meteor") or 0 for _, r in rows]
    x = np.arange(len(rows))
    plt.figure(figsize=(max(8, len(rows) * 0.75), 4))
    plt.bar(x - 0.18, bleu, width=0.36, label="BLEU-4")
    plt.bar(x + 0.18, meteor, width=0.36, label="METEOR")
    plt.xticks(x, labels, rotation=45, ha="right")
    plt.ylabel("score")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "analysis_variations_bleu_meteor.png")
    plt.show()

plot_variation_bleu(rnn_variation_table, lstm_variation_table)

## 11. Qualitative Analysis

In [ ]:
def detail_by_image(result):
    details = result.get("details", []) if result else []
    return {row["image"]: row for row in details}

rnn_details = detail_by_image((rnn_best_eval or {}).get("keras", {}))
lstm_details = detail_by_image((lstm_best_eval or {}).get("keras", {}))
common_images = sorted(set(rnn_details) & set(lstm_details))
print("Common evaluated images:", len(common_images))

In [ ]:
def choose_qualitative_examples(common, k=10):
    scored = []
    for image in common:
        rb = rnn_details[image].get("bleu4", 0)
        lb = lstm_details[image].get("bleu4", 0)
        scored.append((image, (rb + lb) / 2))
    if not scored:
        return []
    scored.sort(key=lambda item: item[1])
    picks = []
    for frac in np.linspace(0, 1, k):
        idx = int(round(frac * (len(scored) - 1)))
        picks.append(scored[idx][0])
    return list(dict.fromkeys(picks))

qual_images = choose_qualitative_examples(common_images, 10)
qual_rows = []
for image in qual_images:
    qual_rows.append({
        "image": image,
        "rnn_bleu4": rnn_details[image].get("bleu4"),
        "lstm_bleu4": lstm_details[image].get("bleu4"),
        "rnn_caption": rnn_details[image].get("candidate"),
        "lstm_caption": lstm_details[image].get("candidate"),
        "ground_truth": rnn_details[image].get("references", [])[:2],
    })
qual_rows

## 12. Analysis 5a: Layers and Hidden State

In [ ]:
def aggregate_by(rows, key):
    groups = {}
    for row in rows:
        groups.setdefault(row[key], []).append(row)
    return {
        k: {
            "mean_bleu4": float(np.mean([r.get("bleu4") or 0 for r in v])),
            "mean_meteor": float(np.mean([r.get("meteor") or 0 for r in v])),
            "mean_time_ms": float(np.mean([r.get("avg_time_ms") or 0 for r in v])),
        }
        for k, v in groups.items()
    }

analysis_5a = {
    "rnn_by_layers": aggregate_by(rnn_variation_table, "layers"),
    "rnn_by_hidden": aggregate_by(rnn_variation_table, "hidden"),
    "lstm_by_layers": aggregate_by(lstm_variation_table, "layers"),
    "lstm_by_hidden": aggregate_by(lstm_variation_table, "hidden"),
}
analysis_5a

Catatan analisis: isi kesimpulan akhir setelah eksperimen dijalankan. Umumnya hidden state lebih besar memberi kapasitas representasi lebih baik tetapi inference lebih lambat; layer tambahan bisa membantu konteks sekuensial, namun mudah overfit pada Flickr8k jika epoch/data terbatas.

## 13. Analysis 5b: Keras vs Scratch

In [ ]:
def keras_scratch_comparison(eval_payload, label):
    if not eval_payload:
        return None
    k = eval_payload.get("keras", {}).get("metrics", {})
    s = eval_payload.get("scratch", {}).get("metrics", {})
    return {
        "decoder": label,
        "keras_bleu4": k.get("bleu4"),
        "scratch_bleu4": s.get("bleu4"),
        "keras_meteor": k.get("meteor"),
        "scratch_meteor": s.get("meteor"),
        "keras_avg_time_ms": k.get("avg_time_ms"),
        "scratch_avg_time_ms": s.get("avg_time_ms"),
    }

comparison_keras_scratch = [
    keras_scratch_comparison(rnn_best_eval, "RNN"),
    keras_scratch_comparison(lstm_best_eval, "LSTM"),
]
[row for row in comparison_keras_scratch if row]

Catatan analisis: scratch memakai operasi NumPy dan loop Python sehingga biasanya lebih lambat dari Keras/TensorFlow. Score seharusnya dekat jika bobot dan urutan operasi sama; selisih kecil dapat muncul dari detail numerik, masking, atau perbedaan decoding step.

## 14. Analysis 5c: RNN vs LSTM

In [ ]:
def compare_rnn_lstm_details():
    rows = []
    for image in common_images:
        r = rnn_details[image]
        l = lstm_details[image]
        rows.append({
            "image": image,
            "rnn_bleu4": r.get("bleu4", 0),
            "lstm_bleu4": l.get("bleu4", 0),
            "rnn_meteor": r.get("meteor", 0),
            "lstm_meteor": l.get("meteor", 0),
        })
    return rows

rnn_lstm_rows = compare_rnn_lstm_details()
rnn_lstm_summary = {
    "n_common": len(rnn_lstm_rows),
    "rnn_bleu4": float(np.mean([r["rnn_bleu4"] for r in rnn_lstm_rows])) if rnn_lstm_rows else None,
    "lstm_bleu4": float(np.mean([r["lstm_bleu4"] for r in rnn_lstm_rows])) if rnn_lstm_rows else None,
    "rnn_meteor": float(np.mean([r["rnn_meteor"] for r in rnn_lstm_rows])) if rnn_lstm_rows else None,
    "lstm_meteor": float(np.mean([r["lstm_meteor"] for r in rnn_lstm_rows])) if rnn_lstm_rows else None,
}
rnn_lstm_summary

Catatan analisis: LSTM biasanya lebih stabil untuk caption karena gate input/forget/output membantu mempertahankan informasi jangka panjang dan mengurangi vanishing gradient. RNN sederhana lebih ringan, tetapi konteks awal caption cenderung lebih cepat hilang pada sekuens yang panjang.

## 15. Max Caption Length Study

In [ ]:
def best_overall_for_length_study():
    candidates = []
    for label, best, payload in [("rnn", best_rnn, rnn_best_eval), ("lstm", best_lstm, lstm_best_eval)]:
        metrics = (payload or {}).get("keras", {}).get("metrics", {})
        if best and metrics:
            candidates.append((label, best["experiment"], metrics.get("bleu4") or 0, metrics.get("avg_time_ms") or float("inf")))
    if not candidates:
        return None
    return max(candidates, key=lambda item: (item[2], -item[3]))

best_length_model = best_overall_for_length_study()
best_length_model

In [ ]:
caption_lengths = [15, 25, MAX_SEQ_LEN]
length_bleu_results = {}
if RUN_FULL_EVAL and best_length_model and eval_names:
    decoder_type, exp_id, _, _ = best_length_model
    model = load_keras_caption_model(exp_id)
    for length in caption_lengths:
        res = evaluate_with_time(model, decoder_type, "keras", eval_names, max_len=length)
        length_bleu_results[length] = res["metrics"]["bleu4"]
    write_json(EVAL_DIR / "caption_length_study.json", length_bleu_results)
else:
    length_bleu_results = read_json(EVAL_DIR / "caption_length_study.json", {})

length_bleu_results

In [ ]:
if length_bleu_results:
    xs = [int(k) for k in length_bleu_results.keys()]
    ys = [length_bleu_results[str(k)] if str(k) in length_bleu_results else length_bleu_results[k] for k in xs]
    plt.figure(figsize=(5, 3))
    plt.plot(xs, ys, marker="o")
    plt.xlabel("max caption length")
    plt.ylabel("BLEU-4")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "caption_length_bleu.png")
    plt.show()

## 16. Bonus: Beam Search

In [ ]:
def beam_comparison(best_row, decoder_type, beam_sizes=(1, 3, 5)):
    if not best_row or not eval_names:
        return {}
    exp_id = best_row["experiment"]
    scratch = load_scratch_caption_model(exp_id, decoder_type)
    rows = {}
    for beam in beam_sizes:
        decoding = "greedy" if beam == 1 else "beam"
        res = evaluate_with_time(scratch, decoder_type, "scratch", eval_names, decoding=decoding, beam_size=beam)
        rows[f"beam_{beam}"] = res["metrics"]
    return rows

if RUN_FULL_EVAL:
    rnn_beam = beam_comparison(best_rnn, "rnn")
    lstm_beam = beam_comparison(best_lstm, "lstm")
    write_json(EVAL_DIR / "beam_search.json", {"rnn": rnn_beam, "lstm": lstm_beam})
else:
    beam_payload = read_json(EVAL_DIR / "beam_search.json", {})
    rnn_beam = beam_payload.get("rnn", {})
    lstm_beam = beam_payload.get("lstm", {})

rnn_beam, lstm_beam

In [ ]:
def sample_beam_captions(best_row, decoder_type, image_name=None):
    if not best_row or not eval_names:
        return {}
    image_name = image_name or eval_names[0]
    feats = load_features(str(FEATURES_PATH))
    scratch = load_scratch_caption_model(best_row["experiment"], decoder_type)
    feat = np.asarray(feats[image_name], dtype=np.float32)
    return {
        "image": image_name,
        "greedy": scratch.generate_caption(feat, max_len=MAX_SEQ_LEN),
        "beam3": scratch.generate_caption_beam(feat, max_len=MAX_SEQ_LEN, beam_size=3),
        "beam5": scratch.generate_caption_beam(feat, max_len=MAX_SEQ_LEN, beam_size=5),
        "references": captions.get(image_name, []),
    }

sample_beam_rnn = sample_beam_captions(best_rnn, "rnn") if best_rnn and eval_names else {}
sample_beam_lstm = sample_beam_captions(best_lstm, "lstm") if best_lstm and eval_names else {}
sample_beam_rnn, sample_beam_lstm

## 17. Bonus: Batch Inference from Scratch

In [ ]:
def generate_captions_batch_scratch(model: CaptionModel, feature_batch: np.ndarray, max_len=MAX_SEQ_LEN):
    pad_idx = model.word2idx["<pad>"]
    start_idx = model.word2idx["<start>"]
    end_idx = model.word2idx["<end>"]
    batch = feature_batch.shape[0]
    hidden_size = model.decoder.cells[0].hidden_size
    x_img = model.image_projection.forward(feature_batch.astype(np.float32))

    if model.decoder_type == "rnn":
        h = [np.zeros((batch, hidden_size), dtype=np.float32) for _ in range(model.n_layers)]
        if model.injection_method == "pre":
            _, h = model.decoder.step(x_img, h)
        tokens = np.full(batch, start_idx, dtype=np.int32)
        finished = np.zeros(batch, dtype=bool)
        generated = [[] for _ in range(batch)]
        for _ in range(max_len):
            x = model.embedding.forward(tokens)
            out, h = model.decoder.step(x, h)
            if model.injection_method == "init":
                out = np.concatenate([out, x_img], axis=-1)
            probs = model.output_layer.forward(out)
            tokens = np.argmax(probs, axis=-1).astype(np.int32)
            for i, tok in enumerate(tokens):
                if finished[i]:
                    continue
                if tok in (end_idx, pad_idx):
                    finished[i] = True
                else:
                    generated[i].append(model.idx2word.get(int(tok), "<unk>"))
            if finished.all():
                break
        return [" ".join(words) for words in generated]

    h = [np.zeros((batch, hidden_size), dtype=np.float32) for _ in range(model.n_layers)]
    c = [np.zeros((batch, hidden_size), dtype=np.float32) for _ in range(model.n_layers)]
    if model.injection_method == "pre":
        _, h, c = model.decoder.step(x_img, h, c)
    tokens = np.full(batch, start_idx, dtype=np.int32)
    finished = np.zeros(batch, dtype=bool)
    generated = [[] for _ in range(batch)]
    for _ in range(max_len):
        x = model.embedding.forward(tokens)
        out, h, c = model.decoder.step(x, h, c)
        if model.injection_method == "init":
            out = np.concatenate([out, x_img], axis=-1)
        probs = model.output_layer.forward(out)
        tokens = np.argmax(probs, axis=-1).astype(np.int32)
        for i, tok in enumerate(tokens):
            if finished[i]:
                continue
            if tok in (end_idx, pad_idx):
                finished[i] = True
            else:
                generated[i].append(model.idx2word.get(int(tok), "<unk>"))
        if finished.all():
            break
    return [" ".join(words) for words in generated]

In [ ]:
def batch_benchmark(best_row, decoder_type):
    if not best_row or not eval_names:
        return {}
    names = eval_names[:N_BATCH]
    feats = load_features(str(FEATURES_PATH))
    batch_feats = np.stack([feats[name] for name in names]).astype(np.float32)
    model = load_scratch_caption_model(best_row["experiment"], decoder_type)

    t0 = time.perf_counter()
    single_caps = [model.generate_caption(feat, MAX_SEQ_LEN) for feat in batch_feats]
    t_single = time.perf_counter() - t0

    t0 = time.perf_counter()
    batch_caps = generate_captions_batch_scratch(model, batch_feats, MAX_SEQ_LEN)
    t_batch = time.perf_counter() - t0

    return {
        "n": len(names),
        "single_total_s": t_single,
        "batch_total_s": t_batch,
        "single_avg_ms": 1000 * t_single / max(len(names), 1),
        "batch_avg_ms": 1000 * t_batch / max(len(names), 1),
        "match_rate": float(np.mean([a == b for a, b in zip(single_caps, batch_caps)])) if names else 0,
    }

batch_rnn = batch_benchmark(best_rnn, "rnn") if best_rnn and eval_names else {}
batch_lstm = batch_benchmark(best_lstm, "lstm") if best_lstm and eval_names else {}
batch_rnn, batch_lstm

## 18. Bonus: Init-Inject Architecture

In [ ]:
def init_inject_configs_from_best():
    configs = []
    if best_rnn:
        configs.append(make_config("rnn", best_rnn["layers"], best_rnn["hidden"], injection_method="init"))
    if best_lstm:
        configs.append(make_config("lstm", best_lstm["layers"], best_lstm["hidden"], injection_method="init"))
    return configs

init_configs = init_inject_configs_from_best()
[c.experiment_id for c in init_configs]

In [ ]:
init_trained, init_skipped = train_grid(init_configs, RUN_TRAIN_INIT_INJECT)
{"trained": init_trained, "available_or_skipped": init_skipped}

In [ ]:
def compare_pre_vs_init(decoder_type, best_pre_row):
    if not best_pre_row or not eval_names:
        return {}
    init_config = make_config(decoder_type, best_pre_row["layers"], best_pre_row["hidden"], injection_method="init")
    init_path = MODELS_DIR / init_config.experiment_id / "model.keras"
    if not init_path.exists():
        return {"missing": str(init_path)}
    pre_model = load_keras_caption_model(best_pre_row["experiment"])
    init_model = load_keras_caption_model(init_config.experiment_id)
    return {
        "pre": evaluate_with_time(pre_model, decoder_type, "keras", eval_names, injection="pre")["metrics"],
        "init": evaluate_with_time(init_model, decoder_type, "keras", eval_names, injection="init")["metrics"],
    }

if RUN_FULL_EVAL:
    init_inject_results = {
        "rnn": compare_pre_vs_init("rnn", best_rnn),
        "lstm": compare_pre_vs_init("lstm", best_lstm),
    }
    write_json(EVAL_DIR / "init_inject_comparison.json", init_inject_results)
else:
    init_inject_results = read_json(EVAL_DIR / "init_inject_comparison.json", {})

init_inject_results

## 19. Bonus: Scratch Batch Sanity

In [ ]:
def check_rnn_batch_sanity():
    rng = np.random.default_rng(0)
    batch, seq_len, input_dim, hidden = 2, 5, 4, 3
    x = rng.normal(size=(batch, seq_len, input_dim)).astype(np.float32)
    cell = SimpleRNNCell()
    cell.W_x = rng.normal(size=(input_dim, hidden)).astype(np.float32)
    cell.W_h = rng.normal(size=(hidden, hidden)).astype(np.float32)
    cell.b = rng.normal(size=(hidden,)).astype(np.float32)
    decoder = SimpleRNNDecoder([cell])
    outputs, h = decoder.forward(x)
    assert outputs.shape == (batch, seq_len, hidden), outputs.shape
    return outputs.shape


def check_lstm_batch_sanity():
    rng = np.random.default_rng(1)
    batch, seq_len, input_dim, hidden = 2, 6, 4, 3
    x = rng.normal(size=(batch, seq_len, input_dim)).astype(np.float32)
    cell = LSTMCell()
    cell.set_weights(
        rng.normal(size=(input_dim, 4 * hidden)).astype(np.float32),
        rng.normal(size=(hidden, 4 * hidden)).astype(np.float32),
        rng.normal(size=(4 * hidden,)).astype(np.float32),
    )
    decoder = LSTMDecoder([cell])
    outputs, h, c = decoder.forward(x)
    assert outputs.shape == (batch, seq_len, hidden), outputs.shape
    return outputs.shape

check_rnn_batch_sanity(), check_lstm_batch_sanity()

## 20. Bonus: Backward Propagation Sanity

In [ ]:
def relative_error(a, b, eps=1e-8):
    denom = np.maximum(eps, np.maximum(np.abs(a), np.abs(b)))
    return float(np.max(np.abs(a - b) / denom))


def rnn_cell_backward_sanity():
    rng = np.random.default_rng(0)
    x = rng.normal(size=(1, 3)).astype(np.float32)
    h_prev = rng.normal(size=(1, 2)).astype(np.float32)
    cell = SimpleRNNCell()
    cell.W_x = rng.normal(size=(3, 2)).astype(np.float32)
    cell.W_h = rng.normal(size=(2, 2)).astype(np.float32)
    cell.b = rng.normal(size=(2,)).astype(np.float32)
    h, cache = cell.forward_with_cache(x, h_prev)
    _, _, gWx, gWh, gb = cell.backward(np.ones_like(h), cache)
    return {"gWx": gWx.shape, "gWh": gWh.shape, "gb": gb.shape}


def lstm_cell_backward_sanity():
    rng = np.random.default_rng(1)
    x = rng.normal(size=(1, 3)).astype(np.float32)
    h_prev = rng.normal(size=(1, 2)).astype(np.float32)
    c_prev = rng.normal(size=(1, 2)).astype(np.float32)
    cell = LSTMCell()
    cell.set_weights(
        rng.normal(size=(3, 8)).astype(np.float32),
        rng.normal(size=(2, 8)).astype(np.float32),
        rng.normal(size=(8,)).astype(np.float32),
    )
    h, _, cache = cell.forward_with_cache(x, h_prev, c_prev)
    _, _, _, gWx, gWh, gb = cell.backward(np.ones_like(h), np.zeros_like(h), cache)
    return {"gWx": gWx.shape, "gWh": gWh.shape, "gb": gb.shape}

rnn_cell_backward_sanity(), lstm_cell_backward_sanity()

In [ ]:
def finite_difference_rnn_Wx_error():
    rng = np.random.default_rng(7)
    x = rng.normal(size=(1, 2)).astype(np.float32)
    h_prev = rng.normal(size=(1, 2)).astype(np.float32)
    cell = SimpleRNNCell()
    cell.W_x = rng.normal(size=(2, 2)).astype(np.float32)
    cell.W_h = rng.normal(size=(2, 2)).astype(np.float32)
    cell.b = rng.normal(size=(2,)).astype(np.float32)
    h, cache = cell.forward_with_cache(x, h_prev)
    _, _, gWx, _, _ = cell.backward(np.ones_like(h), cache)
    grad_num = np.zeros_like(cell.W_x)
    eps = 1e-4
    for i in range(cell.W_x.shape[0]):
        for j in range(cell.W_x.shape[1]):
            orig = cell.W_x[i, j]
            cell.W_x[i, j] = orig + eps
            loss_pos = float(np.sum(cell.forward_with_cache(x, h_prev)[0]))
            cell.W_x[i, j] = orig - eps
            loss_neg = float(np.sum(cell.forward_with_cache(x, h_prev)[0]))
            grad_num[i, j] = (loss_pos - loss_neg) / (2 * eps)
            cell.W_x[i, j] = orig
    return relative_error(gWx, grad_num)

finite_difference_rnn_Wx_error()

## 21. Export Summary Artifacts

In [ ]:
summary_payload = {
    "best_rnn": best_rnn,
    "best_lstm": best_lstm,
    "rnn_variations": rnn_variation_table,
    "lstm_variations": lstm_variation_table,
    "keras_vs_scratch": [row for row in comparison_keras_scratch if row],
    "rnn_vs_lstm": rnn_lstm_summary,
    "caption_length": length_bleu_results,
    "beam": {"rnn": rnn_beam, "lstm": lstm_beam},
    "batch": {"rnn": batch_rnn, "lstm": batch_lstm},
    "init_inject": init_inject_results,
    "qualitative_examples": qual_rows,
}
write_json(WORKING_DIR / "notebook_summary.json", summary_payload)
WORKING_DIR / "notebook_summary.json"

## 22. Final Checklist

- CNN encoder frozen menghasilkan `features.npy`.
- Caption preprocessing menghasilkan `vocab.json` dan `splits.json`.
- RNN dan LSTM masing-masing memiliki 6 variasi pre-inject.
- Best model dievaluasi untuk Keras dan scratch.
- Analisis variasi layer/hidden, Keras vs scratch, RNN vs LSTM, dan panjang caption tersedia.
- Bonus beam search, batch inference, init-inject, dan backward sanity tersedia di notebook ini.